# Tutorial 01: Beginner Interactive Phenotyping

**About this tutorial**
- **Time:** ~15 minutes
- **Prerequisites:** None (widgets-first)
- **Key ideas:** Interactive widgets, minimal code, optional toy dataset or your own images

Welcome! This notebook provides an easy-to-use, interactive way to explore the Phenotyping API **no programming experience necessary**.

To keep things simple, many technical parameters are hidden, letting you focus on the essential choices to get started quickly. Most steps work "out of the box" using recommended settings. If you have custom datasets that need more control, check out the other notebooks or dive into the documentation.

By default, this notebook prepares a toy dataset (E. coli bacteria). You can also bring your own data, masks, and metadata (CSV file).

**Author:** Aitor González-Marfil ([@AAitorG](https://github.com/AAitorG))

In [ ]:
# @title Run this cell to install the required libraries (Colab only)
def in_colab():
    try:
        import google.colab

        print(google.colab.__file__)
        return True
    except ImportError:
        return False


if in_colab():
    import os

    # Install only in Colab
    if not os.path.exists("PhenoMe"):
        !git clone https://github.com/AAitorG/PhenoMe.git
    %cd PhenoMe/
    # Install dependencies required by the notebook and the package
    %pip install -q -e .


## Step 1: Get Ready
Run the cell below to load the necessary libraries and initialize the interface.

In [ ]:
# @title Run this cell to setup the environment
import os
import sys
import zipfile
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import requests
import torch
from IPython.display import clear_output, display

# Enable custom widget manager for Google Colab
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

# Add src/ to path so `phenome` resolves without pip install
sys.path.append(os.path.abspath("../.."))

from phenome import PhenoMe, load_dinov2_model, make_dataframe_metadata_fn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Global state
pheno = None
file_df = None
dist_results = None
metadata_df = None
model_wrapper = None
using_toy_dataset = False


def setup_ecoli_dataset(output_dir="./toy_dataset"):
    """
    Download and extract E. coli dataset from SharePoint.

    Args:
        output_dir: Directory to save the dataset

    Returns:
        Path to the dataset directory
    """
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True, parents=True)

    # SharePoint link with download=1
    url = "https://upvehueus-my.sharepoint.com/:u:/g/personal/aitor_gonzalezm_ehu_eus/IQCBYEL7E_-NTZZeIsBhwwG8AbI9BucCBAgaPacYqXjDsA4?download=1"
    zip_path = output_path / "ecoli_dataset.zip"

    # Improved check: look for Ecoli_images anywhere inside output_path
    existing_images = list(output_path.glob("**/Ecoli_images"))
    if existing_images:
        dataset_root = existing_images[0].parent
        print(f"Dataset already exists at {dataset_root}. Using existing files.")
        return str(dataset_root)

    print("Downloading E. coli dataset (this may take a few minutes)...\n")

    # Setup loading bar
    progress = widgets.IntProgress(
        value=0,
        min=0,
        max=100,
        description="Downloading:",
        bar_style="info",
        orientation="horizontal",
        layout=widgets.Layout(width="100%"),
    )
    display(progress)

    try:
        r = requests.get(url, allow_redirects=True, stream=True)
        total_size = int(r.headers.get("content-length", 0))

        if total_size == 0:
            # Fallback if content-length is not provided
            with open(zip_path, "wb") as f:
                f.write(r.content)
            progress.value = 100
        else:
            progress.max = total_size
            downloaded = 0
            with open(zip_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
                        progress.value = downloaded

        progress.bar_style = "success"
        progress.description = "Done!"

        print("Extracting dataset...")
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(output_path)

        # Clean up zip
        zip_path.unlink()

        # Find the actual data root after extraction
        extracted_images = list(output_path.glob("**/Ecoli_images"))
        if extracted_images:
            dataset_root = extracted_images[0].parent
        else:
            # Fallback to finding any subdir if Ecoli_images is not found
            subdirs = [d for d in output_path.iterdir() if d.is_dir()]
            dataset_root = subdirs[0] if subdirs else output_path

        print(f"Dataset ready at {dataset_root}")
        return str(dataset_root)
    except Exception as e:
        progress.bar_style = "danger"
        progress.description = "Error!"
        print(f"Error setting up dataset: {e}")
        return None

## Step 1.1: Choose Your AI Observer

Select an AI model from the list.
Currently, the models available are DINOv2 of different sizes. The larger the model, the better it detects subtle biological details, but it will take longer to run.

In [ ]:
# @title (Run this cell) Select and load a DINOv2 Model
# DINOv2 model selection and loading
model_options = {
    "Small (ViT-S/14)": ("dinov2_vits14_reg", "<b>Small (ViT-S/14)</b>: Fast enough for quick tests on regular computers."),
    "Base (ViT-B/14)": ("dinov2_vitb14_reg", "<b>Base (ViT-B/14)</b>: A balanced middle ground."),
    "Large (ViT-L/14)": ("dinov2_vitl14_reg", "<b>Large (ViT-L/14)</b>: Better at capturing fine details, but slower."),
    "Giant (ViT-g/14)": ("dinov2_vitg14_reg", "<b>Giant (ViT-g/14)</b>: The most detailed observer, best to run on computers with dedicated graphics cards (GPUs)."),
}

model_dropdown = widgets.Dropdown(
    options=list(model_options.keys()),
    value="Small (ViT-S/14)",
    description="Model Size:",
    style={"description_width": "initial"},
)

model_description = widgets.HTML(
    value=model_options["Small (ViT-S/14)"][1], 
    layout=widgets.Layout(margin="0 0 10px 0")
)

def on_model_change(change):
    model_description.value = model_options[change["new"]][1]

model_dropdown.observe(on_model_change, names="value")

load_model_button = widgets.Button(description="Load Model", button_style="primary")
model_output = widgets.Output()


def on_load_model_clicked(b):
    global model_wrapper, pheno
    with model_output:
        clear_output()
        try:
            model_name = model_options[model_dropdown.value][0]
            print(f"Loading {model_dropdown.value} ({model_name})...")
            model_wrapper = load_dinov2_model(model_name=model_name, device=device)
            pheno = PhenoMe(device=device, seed=42)
            print("✓ Model loaded successfully!")
        except Exception as e:
            print(f"Error loading model: {e}")
            import traceback

            traceback.print_exc()


load_model_button.on_click(on_load_model_clicked)
display(widgets.VBox([model_dropdown, model_description, load_model_button, model_output]))

## Step 2: Load Your Images

- **Use Toy Dataset**: Automatically downloads *E. coli* dataset from Spahn, C., et al., Nat Commun 2025, https://doi.org/10.1038/s41467-025-58723-4. Cite the publication if you use the dataset.

- **Use Custom Dataset**: Provide image directory and optional mask directory

**If you use a CSV:** Only images that appear **both** in your images directory (after PhenoMe scans it) **and** in the CSV’s *filename* column are loaded. Extra CSV rows or extra files on disk without a matching row are not combined into one list—see the [Experiment Details guide](../../docs/src/content/docs/guides/experiment-details.md#2-using-a-csv-or-spreadsheet).

In [ ]:
# @title (Run this cell) Select and load a dataset
dataset_mode = widgets.RadioButtons(
    options=["Use Toy Dataset (Default)", "Use Custom Dataset"],
    value="Use Toy Dataset (Default)",
    description="Dataset Mode:",
    style={"description_width": "initial"},
)

csv_path_input = widgets.Text(
    value="",
    description="CSV Path (optional):",
    placeholder="(Optional) Enter path to your metadata CSV file",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%"),
)

data_path_input = widgets.Text(
    value="",
    description="Images Directory:",
    placeholder="Enter path to directory containing images",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%"),
)

mask_path_input = widgets.Text(
    value="",
    description="Masks Directory:",
    placeholder="(Optional) Enter path to directory containing masks",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%"),
)

load_button = widgets.Button(description="Load Dataset", button_style="primary")
output = widgets.Output()


def on_load_clicked(b):
    global file_df, metadata_df, using_toy_dataset
    if pheno is None:
        with output:
            print("Error: Please load a model first!")
        return

    with output:
        clear_output()
        try:
            if dataset_mode.value == "Use Toy Dataset (Default)":
                using_toy_dataset = True
                print("Setting up toy dataset...")
                dataset_root = setup_ecoli_dataset()
                if not dataset_root:
                    print("Error setting up toy dataset!")
                    return

                dataset_root_path = Path(dataset_root)

                # The root is now correctly identified by setup_ecoli_dataset
                images_dir = dataset_root_path / "Ecoli_images"
                masks_dir = dataset_root_path / "Ecoli_masks"
                metadata_csv = dataset_root_path / "metadata.csv"

                # Fallback if structure is flat or different
                if not images_dir.exists():
                    images_dir = dataset_root_path
                    masks_dir = dataset_root_path
                    metadata_csv = dataset_root_path / "metadata.csv"

                if metadata_csv.exists():
                    print(f"Using existing metadata.csv in {dataset_root_path}")
                    metadata_df = pd.read_csv(metadata_csv)
                    metadata_fn = make_dataframe_metadata_fn(metadata_df)
                else:
                    print("Warning: metadata.csv not found. No metadata will be extracted.")
                    metadata_fn = None

                print(f"\nScanning images in {images_dir}...")
                file_df = pheno.find_files(
                    str(images_dir),
                    extensions=[".tif"],
                    metadata_fn=metadata_fn,
                    mask_dir=str(masks_dir) if masks_dir.exists() else None,
                )
                # Set mask path input for toy dataset
                if masks_dir.exists():
                    mask_path_input.value = str(masks_dir)
                    print(f"Masks found at: {masks_dir}")
                else:
                    mask_path_input.value = str(images_dir)
                    print(f"Masks directory set to same as images: {images_dir}")

                print("Done!")

            else:
                using_toy_dataset = False
                # Clear mask path when switching to custom dataset
                mask_path_input.value = ""
                if not data_path_input.value:
                    print("Error: Please provide an images directory path!")
                    return

                use_metadata = bool(csv_path_input.value.strip())
                if use_metadata:
                    print(f"Loading metadata from {csv_path_input.value}...")
                    metadata_df = pd.read_csv(csv_path_input.value)
                    print(f"Loaded metadata with columns: {list(metadata_df.columns)}")
                    metadata_fn = make_dataframe_metadata_fn(metadata_df)
                else:
                    metadata_df = None
                    metadata_fn = None
                    print("No CSV metadata file provided. Proceeding to scan for images only.")

                print(f"\nScanning images in {data_path_input.value}...")
                mask_dir_val = mask_path_input.value.strip() or None
                file_df = pheno.find_files(
                    data_path_input.value, metadata_fn=metadata_fn, mask_dir=mask_dir_val
                )
            if file_df is not None:
                display(file_df.head())
                print(f"\nFound {len(file_df)} files.")
                print(
                    f"Available metadata columns: {[c for c in file_df.columns if c != 'file_path']}"
                )
            else:
                print("No files found!")
        except Exception as e:
            print(f"Error loading dataset: {e}")
            import traceback

            traceback.print_exc()


load_button.on_click(on_load_clicked)


def update_ui(change):
    if dataset_mode.value == "Use Custom Dataset":
        csv_path_input.layout.display = "flex"
        data_path_input.layout.display = "flex"
        mask_path_input.layout.display = "flex"
        # Clear the mask path when switching to custom dataset through the UI as well
        mask_path_input.value = ""
    else:
        csv_path_input.layout.display = "none"
        data_path_input.layout.display = "none"
        mask_path_input.layout.display = "none"


dataset_mode.observe(update_ui, names="value")
update_ui(None)

display(
    widgets.VBox(
        [dataset_mode, csv_path_input, data_path_input, mask_path_input, load_button, output]
    )
)

## Step 3: Process the images with an AI model

Process each image with the AI model selected in Step 1.1 to obtain the embeddings. This may take a few minutes, depending on the dataset size.
If the images are multichannel, there are two ways of obtaining the encoding. If not, leave with the default value:

- **split**: Treats each channel individually but groups the final summary. Useful when colors measure totally different biological features.
- **combined**: Combines all colors as a human eye sees a standard photo (RGB). Good for general appearance.

In [ ]:
# @title (Run this cell) Select the channel mode and process the dataset
# Channel mode selection
channel_mode_options = {
    'split': "<b>'split'</b>: Treats each channel individually but groups the final summary. Useful when colors measure totally different biological features.",
    'combined': "<b>'combined'</b>: Combines all colors as a human eye sees a standard photo (RGB). Good for general appearance.",
}

channel_mode_dropdown = widgets.Dropdown(
    options=list(channel_mode_options.keys()),
    value='split',
    description='Channel Mode:',
    style={'description_width': 'initial'},
    tooltip="'split': Process each channel independently. 'combined': Combine all channels into RGB.",
)

channel_mode_description = widgets.HTML(
    value=channel_mode_options['split'], 
    layout=widgets.Layout(margin='0 0 10px 0')
)

def on_channel_mode_change(change):
    channel_mode_description.value = channel_mode_options[change['new']]

channel_mode_dropdown.observe(on_channel_mode_change, names='value')

process_button = widgets.Button(description='Process Images', button_style='success')
process_output = widgets.Output()


def on_process_clicked(b):
    if file_df is None or pheno is None:
        with process_output:
            print('Error: Load data first!')
        return

    with process_output:
        clear_output()
        channel_mode = channel_mode_dropdown.value
        preprocessing_fn = None

        print(f'Channel mode: {channel_mode}')

        print('\nExtracting embeddings...')
        pheno.process_images(
            model_wrapper,
            num_workers=2,
            batch_size=128,
            channel_mode=channel_mode,
            preprocessing_fn=preprocessing_fn,
            resize_size=224,
            pad_size=168 if using_toy_dataset else None,
        )
        print('Done!')


process_button.on_click(on_process_clicked)
display(widgets.VBox([channel_mode_dropdown, channel_mode_description, process_button, process_output]))

## Step 3.1: Extract Physical Traits from Images

Compute object and image features like Area (size), Eccentricity (shape), and Intensity (brightness) from your images using masks. These will allow the connection of AI's abstract representation to measurable biology.

- **Basic**: Essential intensity stats (mean, std) and mask shape properties (area, perimeter, eccentricity, solidity). Recommended for most users.
- **Standard**: Comprehensive set including all shape, intensity stats, and concentric ring features.
- **Complete**: Maximum features including texture (GLCM), quality metrics (blur, entropy), and advanced shape descriptors.
- **Shape**: Mask shape properties only (area, eccentricity, axis lengths, etc.). Requires masks.
- **Intensity**: Image intensity statistics only (mean, std, min, max). Works without masks.
- **None**: No preset selected.

In [ ]:
# @title (Run this cell) Select a property preset and compute properties

# Preset options with descriptions
preset_info = {
    "basic": {
        "label": "Basic",
        "description": "<b>Basic</b>: Essential intensity stats (mean, std) and mask shape properties (area, perimeter, eccentricity, solidity). Recommended for most users.",
    },
    "standard": {
        "label": "Standard",
        "description": "<b>Standard</b>: Comprehensive set including all shape, intensity stats, and concentric ring features.",
    },
    "complete": {
        "label": "Complete",
        "description": "<b>Complete</b>: Maximum features including texture (GLCM), quality metrics (blur, entropy), and advanced shape descriptors.",
    },
    "shape": {
        "label": "Shape",
        "description": "<b>Shape</b>: Mask shape properties only (area, eccentricity, axis lengths, etc.). Requires masks.",
    },
    "intensity": {
        "label": "Intensity",
        "description": "<b>Intensity</b>: Image intensity statistics only (mean, std, min, max). Works without masks.",
    },
    "none": {"label": "None", "description": "<b>None</b>: No preset selected."},
}

preset_dropdown = widgets.Dropdown(
    options=[(info["label"], name) for name, info in preset_info.items()],
    value="basic",
    description="Property Preset:",
    style={"description_width": "initial"},
)

preset_description = widgets.HTML(
    value=preset_info["basic"]["description"], layout=widgets.Layout(margin="0 0 10px 0")
)


def on_preset_change(change):
    preset_description.value = preset_info[change["new"]]["description"]


preset_dropdown.observe(on_preset_change, names="value")

compute_button = widgets.Button(description="Compute Properties", button_style="success")
compute_output = widgets.Output()


def on_compute_clicked(b):
    if pheno is None or file_df is None:
        with compute_output:
            print("Error: Load data first!")
        return

    # Verify that images have been processed (embeddings extracted) first
    embeddings = pheno.results.embeddings
    if embeddings is None or len(embeddings) == 0:
        with compute_output:
            print("Error: Please process images first (Step 3) before computing properties!")
        return

    with compute_output:
        clear_output()
        try:
            preset = preset_dropdown.value
            if preset == "none":
                print("Error: Please select a property preset.")
                return

            print(f"Computing properties using preset: {preset_info[preset]['label']}...")

            # Compute properties
            results_df = pheno.compute_properties(property_preset=preset)

            print(f"Done! Computed properties for {len(results_df)} images.")
            display(results_df.head())
        except Exception as e:
            print(f"Error computing properties: {e}")
            import traceback

            traceback.print_exc()


compute_button.on_click(on_compute_clicked)
display(widgets.VBox([preset_dropdown, preset_description, compute_button, compute_output]))

## Step 4: Interactive Visualization of Latent Space Representations

AI embeddings are processed for dimensionality reduction and displayed in a 2D or 3D space (PCA, t-SNE, or UMAP). 

**Click on any point** in the plot to see the corresponding image showing what that specific cell looks like!

In [ ]:
# @title (Run this cell) Configure and visualize the data manifold.
if file_df is not None and pheno is not None:
    explorer = pheno.create_interactive_explorer()
else:
    print("Run previous steps first (load data and process images).")

## Step 5: Explainability

Compute statistical correlations between the latent space representation and the hand-crafted physical features we measured previously. This will allow you to link physical traits and descriptive properties characterising data distribution.

**Note:** For identifying the exact physical traits that differ most between your experimental groups (e.g., Treated vs. Control), see the advanced comparison tools in [Tutorial 03](03_core_phenotyping_workflow.ipynb) and [Tutorial 04](04_exploratory_analysis_and_explainability.ipynb).

In [ ]:
# @title (Run this cell) Configure and run component–property correlations
dr_method_dropdown = widgets.Dropdown(
    options=[("PCA", "pca"), ("t-SNE", "tsne"), ("UMAP", "umap")],
    value="umap",
    description="Dimensionality reduction method:",
    style={"description_width": "initial"},
)

correlation_options = {
    "pearson": "<b>Pearson</b>: Best for linear relationships (default).",
    "spearman": "<b>Spearman</b>: Robust to outliers, detects non-linear but monotonic relationships.",
    "distance_correlation": "<b>Distance Correlation</b>: Detects complex non-linear relationships (requires dcor library).",
    "mutual_info": "<b>Mutual Information</b>: Detects any kind of statistical dependency.",
}

correlation_method_dropdown = widgets.Dropdown(
    options=[(label.split(":")[0].strip(), name) for name, label in correlation_options.items()],
    value="pearson",
    description="Correlation:",
    style={"description_width": "initial"},
)

correlation_description = widgets.HTML(
    value=correlation_options["pearson"], 
    layout=widgets.Layout(margin="0 0 10px 0")
)

def on_correlation_change(change):
    correlation_description.value = correlation_options[change["new"]]

correlation_method_dropdown.observe(on_correlation_change, names="value")

top_k_input = widgets.IntText(
    value=10, min=1, max=50, description="Top K properties:", style={"description_width": "initial"}
)

explain_button = widgets.Button(description="Compute correlations", button_style="primary")
explain_output = widgets.Output()


def on_explain_clicked(b):
    if file_df is None or pheno is None:
        with explain_output:
            print("Run previous steps first (load data and process images).")
        return

    props = pheno.results.properties
    if props is None or (hasattr(props, "empty") and props.empty):
        with explain_output:
            print(
                "Compute morphological properties first (Step 3.1) to get component–property correlations."
            )
        return

    with explain_output:
        clear_output()
        dr_method = dr_method_dropdown.value
        correlation_method = correlation_method_dropdown.value
        # Use dimensions from explorer if available, otherwise default to 2
        n_components = explorer.dim_toggle.value if "explorer" in globals() and explorer is not None else 2
        top_k = top_k_input.value
        print(
            f"Method: {dr_method}, Correlation: {correlation_method}, n_components={n_components}, top_k={top_k}"
        )

        try:
            corr_results = pheno.compute_component_correlation(
                method=dr_method,
                n_components=n_components,
                source="embeddings",
                top_k=top_k,
                correlation_method=correlation_method,
            )
        except ImportError as e:
            if "dcor" in str(e).lower():
                print("Error: Distance correlation is not installed. Please install it with: pip install dcor")
            else:
                print(f"Error: {e}")
        except Exception as e:
            print(f"Error: {e}")
            import traceback

            traceback.print_exc()


explain_button.on_click(on_explain_clicked)
display(
    widgets.VBox(
        [
            dr_method_dropdown,
            correlation_method_dropdown,
            correlation_description,
            top_k_input,
            explain_button,
            explain_output,
        ]
    )
)

## Next Steps

- [03_core_phenotyping_workflow.ipynb](03_core_phenotyping_workflow.ipynb): Core API tutorial
- [04_exploratory_analysis_and_explainability.ipynb](04_exploratory_analysis_and_explainability.ipynb): Advanced analysis

## Learn More

Now that you've explored the interactive basics, check out the advanced documentation to learn how to scale your analysis:

- **[Experiment Details Pipeline](../../docs/src/content/docs/guides/experiment-details.md)**: Replace manual entry with automatic CSV linkage and learn advanced path parsing for multi-condition screens.
- **[Property interpretation](../../docs/src/content/docs/concepts/property-interpretation.md)**: What each built-in classical property measures.
- **[Select properties](../../docs/src/content/docs/guides/select-properties/index.md)**: Choose `property_preset` bundles or inject your own property functions.